# DATA 605 Homework 5: Spark

## Spark background
* What is Spark?
* Why is Spark a popular framework?
* What is Spark SQL and why does it exist?
* What is a Spark DataFrame, and why is it useful?

* Apache Spark is an open-source, distributed computing framework.
* It is designed to process and analyze very large datasets quickly by dividing the work across multiple machines.
* Spark is popular as:
  * It is very fast compared to Hadoop MapReduce as it uses RAM instead of disk.
  * It can easily scale to huge distributed clusters without much changes in the code.
  * It is very simple to use.
  * Native support for multiple programming languages.
  * Enables programming at a higher level of abstraction.

* Spark SQL is a module in Spark that allows to run querys using SQL syntax.
  * It optimizes the query and runs against individual RDDs.
  * It is natively built into Spark.
  * Works with Hive data.

* Spark Dataframe is similar to a Pandas DataFrame in Python or a SQL table.
  * It is built on topvof RDDs
  * Designed and optimized for distributed tabular computing at scale.
  * It can queried using regular SQL syntax.
  * They are easy to work with and handy at scale.
  * Optimizes operations on DataFrames for better performance.



In [2]:
# install pyspark
!pip install pyspark

I'll be performing analysis on the Heart Disease Dataset to predict Heart Failures.

The objective is to perform data preprocessing using *Spark SQL* and build a machine learning model using *Spark MLLib* that can accurately predict whether a person is likely to experience heart failure.

I have downloaded the dataset from Kaggle.
* Link: https://www.kaggle.com/code/tanmay111999/heart-failure-prediction-cv-score-90-5-models/notebook

In [3]:
# Create a Spark session - required to use Spark DataFrames, SQL, and ML features.
from pyspark.sql import SparkSession

#.builder - starts the configuration process.
#.appName - gives Spark job a name that will used in logs
#.getOrCreate - creates a new Spark session if one doesn't already exist.
spark = SparkSession.builder \
    .appName("DATA605 Spark App") \
    .getOrCreate()

In [4]:
# check if the session was created correctly
print(spark.sparkContext.appName)

DATA605 Spark App


##Explore dataset using Spark

In [5]:
# Load the dataset as a Dataframe using Spark
# file_path is the path the csv file
# header =True indicates that the first row contains column names
# inferSchema=True automatically detects datatypes of the columns
file_path = 'heart.csv'
df = spark.read.csv(file_path, header=True, inferSchema=True)
# print the first 5 rows of the dataset
df.show(5)

+---+---+-------------+---------+-----------+---------+----------+-----+--------------+-------+--------+------------+
|Age|Sex|ChestPainType|RestingBP|Cholesterol|FastingBS|RestingECG|MaxHR|ExerciseAngina|Oldpeak|ST_Slope|HeartDisease|
+---+---+-------------+---------+-----------+---------+----------+-----+--------------+-------+--------+------------+
| 40|  M|          ATA|      140|        289|        0|    Normal|  172|             N|    0.0|      Up|           0|
| 49|  F|          NAP|      160|        180|        0|    Normal|  156|             N|    1.0|    Flat|           1|
| 37|  M|          ATA|      130|        283|        0|        ST|   98|             N|    0.0|      Up|           0|
| 48|  F|          ASY|      138|        214|        0|    Normal|  108|             Y|    1.5|    Flat|           1|
| 54|  M|          NAP|      150|        195|        0|    Normal|  122|             N|    0.0|      Up|           0|
+---+---+-------------+---------+-----------+---------+-

In [6]:
# check the structure and datatype of the dataset.
df.printSchema()

root
 |-- Age: integer (nullable = true)
 |-- Sex: string (nullable = true)
 |-- ChestPainType: string (nullable = true)
 |-- RestingBP: integer (nullable = true)
 |-- Cholesterol: integer (nullable = true)
 |-- FastingBS: integer (nullable = true)
 |-- RestingECG: string (nullable = true)
 |-- MaxHR: integer (nullable = true)
 |-- ExerciseAngina: string (nullable = true)
 |-- Oldpeak: double (nullable = true)
 |-- ST_Slope: string (nullable = true)
 |-- HeartDisease: integer (nullable = true)



* Columns which are integer or double are numerical
* Columns which are string are categorical

In [7]:
# print the number of rows and columns in the DataFrame
print(f"Dataset has {df.count()} rows and {len(df.columns)} columns")

Dataset has 918 rows and 12 columns


In [8]:
# Display statistical summary
df.describe().show()

+-------+------------------+----+-------------+------------------+------------------+-------------------+----------+------------------+--------------+------------------+--------+-------------------+
|summary|               Age| Sex|ChestPainType|         RestingBP|       Cholesterol|          FastingBS|RestingECG|             MaxHR|ExerciseAngina|           Oldpeak|ST_Slope|       HeartDisease|
+-------+------------------+----+-------------+------------------+------------------+-------------------+----------+------------------+--------------+------------------+--------+-------------------+
|  count|               918| 918|          918|               918|               918|                918|       918|               918|           918|               918|     918|                918|
|   mean|53.510893246187365|NULL|         NULL|132.39651416122004| 198.7995642701525|0.23311546840958605|      NULL|136.80936819172112|          NULL|0.8873638344226581|    NULL| 0.5533769063180828|
| std

In [18]:
# check datatype of Age
df.schema['Age'].dataType

IntegerType()

In [19]:
from pyspark.sql.functions import isnan, when, count, col
# check dupliacte rows
df.groupBy(df.columns).agg(count('*').alias('duplicates')).filter(col('duplicates') > 1).show()

+---+---+-------------+---------+-----------+---------+----------+-----+--------------+-------+--------+------------+----------+
|Age|Sex|ChestPainType|RestingBP|Cholesterol|FastingBS|RestingECG|MaxHR|ExerciseAngina|Oldpeak|ST_Slope|HeartDisease|duplicates|
+---+---+-------------+---------+-----------+---------+----------+-----+--------------+-------+--------+------------+----------+
+---+---+-------------+---------+-----------+---------+----------+-----+--------------+-------+--------+------------+----------+



* There are no duplicate rows in this dataset

In [22]:
from pyspark.sql.functions import sum as _sum

# check for Null values
# _sum() function counts the total number of nulls per columns
# .isNull returns True for nulls
# .cast converts boolean to int
# .alias labels the result with the column name
df.select(
    [_sum(col(c) \
          .isNull() \
          .cast("int")) \
          .alias(c) \
          for c in df.columns]
    ).show()

+---+---+-------------+---------+-----------+---------+----------+-----+--------------+-------+--------+------------+
|Age|Sex|ChestPainType|RestingBP|Cholesterol|FastingBS|RestingECG|MaxHR|ExerciseAngina|Oldpeak|ST_Slope|HeartDisease|
+---+---+-------------+---------+-----------+---------+----------+-----+--------------+-------+--------+------------+
|  0|  0|            0|        0|          0|        0|         0|    0|             0|      0|       0|           0|
+---+---+-------------+---------+-----------+---------+----------+-----+--------------+-------+--------+------------+



* There are no missing values in this dataset.

In [17]:
# Check the unique values in each column
# Get the list of columns in the DataFrame
cols = df.columns

# Loop through each column to group by and count unique values
for col in cols:
  # check if the column has categorical values
  if df.schema[col].dataType.simpleString() == 'string':
    # groups the DataFrame by the column and count how many times each value appears
    df.groupBy(col).count().show()

+---+-----+
|Sex|count|
+---+-----+
|  F|  193|
|  M|  725|
+---+-----+

+-------------+-----+
|ChestPainType|count|
+-------------+-----+
|          NAP|  203|
|          ATA|  173|
|           TA|   46|
|          ASY|  496|
+-------------+-----+

+----------+-----+
|RestingECG|count|
+----------+-----+
|       LVH|  188|
|    Normal|  552|
|        ST|  178|
+----------+-----+

+--------------+-----+
|ExerciseAngina|count|
+--------------+-----+
|             Y|  371|
|             N|  547|
+--------------+-----+

+--------+-----+
|ST_Slope|count|
+--------+-----+
|    Flat|  460|
|      Up|  395|
|    Down|   63|
+--------+-----+



### Data Quality Check
* I checked the dataset for duplicate rows, missing values, and inconsistencies in categorical features.
* I found that the dataset is clean
    
  — it contains no duplicate records, no null values, and all categorical columns have correct, typo-free values.
* Since the data is well-structured, I don't have to perform any additional modifications. The dataset is ready for modeling.